# H100 NVL roofline: DSMEM vs non-DSMEM workloads

Inputs: `../workloads/*.yaml` (one manifest per workload), `../results/ceilings.csv` (measured ceilings),
`../results/points.csv` (one row per variant × scope × shape × rep; schema in `../docs/DESIGN.md` §5).

**Setup** (fill in): GPU, driver/CUDA/ncu versions, clock state, repo commit — see `../CLAUDE.md`.

**Definitions**: `y = work / time` (TFLOP/s, or Tops/s for integer workloads); `x_L = work / bytes_L` for
memory level L (HBM for the classic plot). Markers: filled = operator scope (measured), hollow = layer/step
scope, star = reported from the paper. Arrows: non_dsmem → control → dsmem of the same workload.

In [ ]:
import glob, yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
R = Path('../results')
manifests = {m['name']: m for m in (yaml.safe_load(open(f)) for f in sorted(glob.glob('../workloads/*.yaml'))
                                     if not Path(f).name.startswith('_'))}
ceil = pd.read_csv(R / 'ceilings.csv').set_index('name')['value']
pts = pd.read_csv(R / 'points.csv')
pts['work'] = pts['flops'].fillna(0) + pts['int_ops'].fillna(0)
pts['tput'] = pts['work'] / (pts['time_ns'] * 1e-9)
for lvl in ['hbm', 'l2', 'l1', 'dsmem']:
    pts[f'ai_{lvl}'] = pts['work'] / pts[f'{lvl}_bytes']
agg = (pts[pts.rep != -1].groupby(['workload','variant_id','role','scope','shape_id','source'])
         .agg(tput=('tput','median'), tput_lo=('tput', lambda s: s.quantile(.1)), tput_hi=('tput', lambda s: s.quantile(.9)),
              ai_hbm=('ai_hbm','median'), ai_l2=('ai_l2','median'), ai_l1=('ai_l1','median'), ai_dsmem=('ai_dsmem','median'),
              time_ns=('time_ns','median'), hbm_bytes=('hbm_bytes','median'), dsmem_bytes=('dsmem_bytes','median'))
         .reset_index())
PREC_CEIL = {'fp16_tensor': 'fp16_tc_peak', 'bf16_tensor': 'fp16_tc_peak', 'fp8_tensor': 'fp8_tc_peak',
             'fp32_cuda': 'fp32_peak', 'int32': 'int32_peak'}
agg['precision'] = agg.workload.map(lambda w: manifests[w]['precision'])
agg.head()

## Fig. 1 — classic roofline (x = FLOP per HBM byte)

In [ ]:
ROLE_ORDER = {'non_dsmem': 0, 'control': 1, 'dsmem': 2}
MARK = {('operator','measured'): ('o', True), ('layer','measured'): ('o', False), ('operator','reported'): ('*', True)}
def roofline_axes(ax, x):
    for name, lab in [('fp16_tc_peak','FP16 TC'), ('fp8_tc_peak','FP8 TC'), ('fp32_peak','FP32'), ('int32_peak','INT32')]:
        if name in ceil: ax.axhline(ceil[name], color='gray', lw=1); ax.text(x[-1], ceil[name]*1.1, lab, ha='right', fontsize=8)
    bw = ceil['hbm_read_bw']; ax.plot(x, bw * x, color='k', lw=1.2, label=f'HBM {bw/1e12:.2f} TB/s (measured)')
    if 'hbm_bw_datasheet' in ceil: ax.plot(x, ceil['hbm_bw_datasheet'] * x, 'k--', lw=.8, label='HBM datasheet')
    ax.set(xscale='log', yscale='log', xlabel='operational intensity (FLOP / HBM byte)', ylabel='achieved throughput (FLOP/s or op/s)')
x = np.logspace(-2, 4, 200)
fig, ax = plt.subplots(figsize=(9, 6)); roofline_axes(ax, x)
colors = {w: f'C{i%10}' for i, w in enumerate(manifests)}
for (w, scope, shape, src), g in agg.groupby(['workload','scope','shape_id','source']):
    g = g.assign(o=g.role.map(ROLE_ORDER)).sort_values('o')
    mk, filled = MARK.get((scope, src), ('s', True))
    ax.plot(g.ai_hbm, g.tput, '-', color=colors[w], lw=.8, alpha=.6)
    ax.scatter(g.ai_hbm, g.tput, marker=mk, s=60, facecolors=colors[w] if filled else 'none', edgecolors=colors[w],
               label=f'{w} [{shape}]' if scope == 'operator' else None)
    # arrows non_dsmem -> control -> dsmem
    if len(g) > 1:
        for a, b in zip(g.itertuples(), list(g.itertuples())[1:]):
            ax.annotate('', xy=(b.ai_hbm, b.tput), xytext=(a.ai_hbm, a.tput), arrowprops=dict(arrowstyle='->', color=colors[w], lw=1))
# compulsory-traffic ticks from the manifests
ax.legend(fontsize=6, ncol=2, loc='lower right'); plt.tight_layout()

## Fig. 2 — hierarchical roofline (intensity per memory level)
Each panel = one memory level with its measured bandwidth ceiling; the DSMEM variant should move right on the HBM panel and left/down on the on-chip panels.

In [ ]:
levels = [('hbm', 'hbm_read_bw', 'HBM'), ('l2', 'l2_bw', 'L2'), ('l1', 'l1_smem_bw', 'L1/SMEM'), ('dsmem', 'dsmem_bw', 'DSMEM')]
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
for ax, (lvl, cname, lab) in zip(axes, levels):
    for name, l in [('fp16_tc_peak','FP16 TC'), ('fp32_peak','FP32')]:
        if name in ceil: ax.axhline(ceil[name], color='gray', lw=1)
    if cname in ceil: ax.plot(x, ceil[cname] * x, 'k', lw=1.2, label=f'{lab} {ceil[cname]/1e12:.1f} TB/s')
    d = agg[agg.scope == 'operator']
    for w, g in d.groupby('workload'):
        g = g.assign(o=g.role.map(ROLE_ORDER)).sort_values(['shape_id','o'])
        ax.scatter(g[f'ai_{lvl}'], g.tput, color=colors[w], s=40, label=w)
    ax.set(xscale='log', yscale='log', title=lab, xlabel=f'FLOP / {lab} byte'); ax.legend(fontsize=6)
axes[0].set_ylabel('achieved throughput'); plt.tight_layout()

## Fig. 3 — per-workload bars: time, HBM bytes, DSMEM bytes, fraction of layer time in the optimized region

In [ ]:
op = agg[agg.scope == 'operator'].copy()
lay = agg[agg.scope == 'layer'][['workload','variant_id','shape_id','time_ns']].rename(columns={'time_ns': 'layer_time_ns'})
op = op.merge(lay, on=['workload','variant_id','shape_id'], how='left')
op['frac_of_layer'] = op['time_ns'] / op['layer_time_ns']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, col, lab in zip(axes, ['time_ns','hbm_bytes','dsmem_bytes','frac_of_layer'], ['time (ns)','HBM bytes','DSMEM bytes','fraction of layer time']):
    piv = op.pivot_table(index=['workload','shape_id'], columns='role', values=col)
    piv.plot.bar(ax=ax, logy=(col != 'frac_of_layer')); ax.set(title=lab); ax.tick_params(axis='x', labelsize=6)
plt.tight_layout()

## Summary table
Operator scope, one row per workload × shape: speedup and byte reduction of the DSMEM variant over the non-DSMEM one.

In [ ]:
s = op.pivot_table(index=['workload','shape_id'], columns='role', values=['time_ns','hbm_bytes','tput','ai_hbm'])
out = pd.DataFrame({'speedup': s[('time_ns','non_dsmem')] / s[('time_ns','dsmem')],
                    'hbm_bytes_ratio': s[('hbm_bytes','dsmem')] / s[('hbm_bytes','non_dsmem')],
                    'ai_hbm_non_dsmem': s[('ai_hbm','non_dsmem')], 'ai_hbm_dsmem': s[('ai_hbm','dsmem')],
                    'tput_dsmem': s[('tput','dsmem')]})
out.round(3)